# Regression analysis
This notebook demonstrates the regression analysis workflow. It first loads the necessary data, shows the predictor screening (Table S2), performs the linear regression analysis and the final plotting for Fig. 3. In addition it shows the supplementary sensitivity test (Table S3) and the complementary random forest anaylsis (Table S5 and Fig. S11). 

### Note

To replicate the results presented in the publication, it is essential to preprocess the complete ESM output data as described in the README or Method section of the publication.

**Following steps are included in this script:**

1. **Load data**
2. **Regression analysis setup**
3. **Predictor screening (Table S2)**
4. **Performance of main MLR and create Fig. 3**
5. **Sensitvity tests (Table S3) and Figure S1**
6. **Random forest analysis (Table S5 and Fig. S11)**

#### Import Required Libraries

In [ ]:
# ========== Import Required Libraries ==========
import sys
import dask
from dask.diagnostics import ProgressBar
import numpy as np
import pandas as pd

# ========== Configure Paths ==========
# Define the full path to the directories containing utility scripts and configurations
sys.path.append('../../src')
sys.path.append('../../src/data_handling')
sys.path.append('../../src/analysis')
sys.path.append('../../src/visualization')

# Import custom utility functions and configurations
import load_data as load_dat
import process_data as pro_dat
import compute_statistics as comp_stats
import global_maps_and_tabels as glob_map_tab

#import data directory
from config import DATA_DIR

# ========== Define Font Style ==========
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Nimbus Sans'

In [ ]:
import importlib
import colormaps_and_utilities as col_uti
import regression_analysis as reg_analysis
import regression_analysis_rf as rf_analysis

### Step 1: Load data

In [ ]:
importlib.reload(comp_stats)

In [ ]:
#Step 1.2: Load change data
combined_dict, combined_dict_diff, hydro_mask = comp_stats.load_and_combine_regression_data()

In [ ]:
#Step 3.2: Subdivide change data into historical blue and green water regimes
ds_dict_change_sub = {}
ds_dict_change_sub = pro_dat.subdivide_ds_dict(combined_dict['historical'], combined_dict_diff['ssp370_ff-historical'], 
                                                base_id="historical",
                                                change_id="ssp370_ff-historical",
                                                variable="bgws_tran_mean",
                                                base_key="12 model ensemble mean",)

### Step 2: Regression analysis setup

In [ ]:
response = "bgws_tran_mean"
candidate_predictors = [
    "pr_mean",
    "RX1day",
    "RX5day",
    "pr_seasonality",
    
    "vpd_mean",
    "vpd_seasonality",
    
    "tas_mean",
    "rsds_mean",
    "clt_mean",
    
    "mrso_mean",
    "mrsos_mean",
    
    "lai_mean",
    "gpp_mean",
    "wue_mean",
]

In [ ]:
ds = ds_dict_change_sub["ssp370_ff-historical"]["12 model ensemble mean"]

In [ ]:
subdivision_names = list(ds.subdivision.values)

df_blue = reg_analysis.make_regime_dataframe(
    ds, subdivision=subdivision_names[0],
    response=response,
    predictors=candidate_predictors,
)

df_green = reg_analysis.make_regime_dataframe(
    ds, subdivision=subdivision_names[1],
    response=response,
    predictors=candidate_predictors,
)

print("Subdivision names:")
print(subdivision_names)
print()

print("Blue regime shape:", df_blue.shape)
print("Green regime shape:", df_green.shape)
print()

print("Blue preview:")
display(df_blue.head())

print("Green preview:")
display(df_green.head())

In [ ]:
print(df_blue.describe().T[["mean", "std", "min", "max"]])
print(df_green.describe().T[["mean", "std", "min", "max"]])
print("Blue n =", len(df_blue))
print("Green n =", len(df_green))

In [ ]:
df_blue = reg_analysis.make_regime_dataframe_with_coords(
    ds, subdivision=subdivision_names[0],
    response=response,
    predictors=candidate_predictors,
)

df_green = reg_analysis.make_regime_dataframe_with_coords(
    ds, subdivision=subdivision_names[1],
    response=response,
    predictors=candidate_predictors,
)

print(df_blue.head())
print(df_green.head())

In [ ]:
df_blue = reg_analysis.add_spatial_blocks(df_blue, lat_bin=10, lon_bin=20)
df_green = reg_analysis.add_spatial_blocks(df_green, lat_bin=10, lon_bin=20)

print("Blue blocks:", df_blue["spatial_block"].nunique())
print("Green blocks:", df_green["spatial_block"].nunique())

In [ ]:
X_blue = df_blue[candidate_predictors].copy()
y_blue = df_blue[response].copy()
groups_blue = df_blue["spatial_block"].copy()

X_green = df_green[candidate_predictors].copy()
y_green = df_green[response].copy()
groups_green = df_green["spatial_block"].copy()

In [ ]:
print(df_blue["spatial_block"].value_counts().head(10))
print(df_green["spatial_block"].value_counts().head(10))

### Step 3: Predictor screening (Table S2)

In [ ]:
blue_results, blue_coefs = reg_analysis.nested_grouped_elasticnet_screening(
    X_blue, y_blue, groups_blue,
    n_splits_outer=5,
    n_splits_inner=4,
)

In [ ]:
print(blue_results[["fold", "test_r2", "best_alpha", "best_l1_ratio", "n_selected"]])
print()
print("Mean outer test R²:", blue_results["test_r2"].mean())
print("Median outer test R²:", blue_results["test_r2"].median())

In [ ]:
blue_coefs

In [ ]:
blue_selection_freq = (blue_coefs != 0).mean(axis=1).sort_values(ascending=False)
blue_sign_stability = np.sign(blue_coefs.replace(0, np.nan)).mean(axis=1)

blue_summary = pd.DataFrame({
    "selection_frequency": blue_selection_freq,
    "mean_coef": blue_coefs.mean(axis=1),
    "median_coef": blue_coefs.median(axis=1),
    "sign_stability": blue_sign_stability,
}).sort_values(["selection_frequency", "median_coef"], ascending=[False, False])

print(blue_summary)

In [ ]:
blue_results_1se, blue_coefs_1se = reg_analysis.nested_grouped_elasticnet_1se(
    X_blue, y_blue, groups_blue,
    n_splits_outer=5,
    n_splits_inner=4,
)

In [ ]:
print(blue_results_1se[["fold", "test_r2", "chosen_alpha", "chosen_l1_ratio", "n_selected"]])
print()
print("Mean outer test R²:", blue_results_1se["test_r2"].mean())
print("Median outer test R²:", blue_results_1se["test_r2"].median())

In [ ]:
blue_selection_freq_1se = (blue_coefs_1se != 0).mean(axis=1).sort_values(ascending=False)
blue_sign_stability_1se = np.sign(blue_coefs_1se.replace(0, np.nan)).mean(axis=1)

blue_summary_1se = pd.DataFrame({
    "selection_frequency": blue_selection_freq_1se,
    "mean_coef": blue_coefs_1se.mean(axis=1),
    "median_coef": blue_coefs_1se.median(axis=1),
    "sign_stability": blue_sign_stability_1se,
}).sort_values(["selection_frequency", "median_coef"], ascending=[False, False])

print(blue_summary_1se)

In [ ]:
blue_rep_results, blue_rep_coefs, blue_rep_summary = reg_analysis.repeated_grouped_elasticnet_1se(
    X_blue, y_blue, groups_blue,
    n_repeats=40,
    test_size=0.2,
    n_splits_inner=4,
    random_state=42,
)

In [ ]:
print(blue_rep_results[["test_r2", "chosen_alpha", "chosen_l1_ratio", "n_selected"]].describe())
print()
print(blue_rep_summary)

In [ ]:
core_blue = blue_rep_summary[
    (blue_rep_summary["selection_frequency"] >= 0.80) &
    (blue_rep_summary["sign_consistency"].abs() >= 0.90)
]

secondary_blue = blue_rep_summary[
    (blue_rep_summary["selection_frequency"] >= 0.50) &
    (blue_rep_summary["selection_frequency"] < 0.80)
]

weak_blue = blue_rep_summary[
    blue_rep_summary["selection_frequency"] < 0.50
]

print("Core predictors")
print(core_blue)

print("\nSecondary predictors")
print(secondary_blue)

print("\nWeak predictors")
print(weak_blue)

In [ ]:
blue_set_A = [
    "RX1day",
    "clt_mean",
    "mrsos_mean",
    "wue_mean",
    "pr_seasonality",
    "mrso_mean",
    "lai_mean",
]

blue_set_B = blue_set_A + [
    "pr_mean",
    "vpd_mean",
]

blue_set_C = blue_set_A + [
    "pr_mean",
    "vpd_mean",
    "vpd_seasonality",
]

blue_set_D = blue_set_C + [
    "RX5day",
]

blue_set_E = blue_set_D + [
    "rsds_mean",
    "tas_mean",
    "gpp_mean",
]

blue_sets = {
    "A_core": blue_set_A,
    "B_core_plus_pr_vpd": blue_set_B,
    "C_core_plus_all_secondary": blue_set_C,
    "D_plus_RX5day": blue_set_D,
    "all": blue_set_E
}

blue_set_results = []

for name, cols in blue_sets.items():
    res, coefs, summary = reg_analysis.repeated_grouped_elasticnet_1se(
        X_blue[cols], y_blue, groups_blue,
        n_repeats=40,
        test_size=0.2,
        n_splits_inner=4,
        random_state=42,
    )
    
    blue_set_results.append({
        "model": name,
        "n_predictors": len(cols),
        "mean_test_r2": res["test_r2"].mean(),
        "median_test_r2": res["test_r2"].median(),
        "sd_test_r2": res["test_r2"].std(),
        "mean_n_selected": res["n_selected"].mean(),
    })

blue_set_results = pd.DataFrame(blue_set_results).sort_values(
    ["mean_test_r2", "median_test_r2"], ascending=False
)

print(blue_set_results)

In [ ]:
green_rep_results, green_rep_coefs, green_rep_summary = reg_analysis.repeated_grouped_elasticnet_1se(
    X_green, y_green, groups_green,
    n_repeats=40,
    test_size=0.2,
    n_splits_inner=4,
    random_state=42,
)

In [ ]:
core_green = green_rep_summary[
    (green_rep_summary["selection_frequency"] >= 0.80) &
    (green_rep_summary["sign_consistency"].abs() >= 0.90)
]

secondary_green = green_rep_summary[
    (green_rep_summary["selection_frequency"] >= 0.50) &
    (green_rep_summary["selection_frequency"] < 0.80)
]

weak_green = green_rep_summary[
    green_rep_summary["selection_frequency"] < 0.50
]

print("Core predictors")
print(core_green)

print("\nSecondary predictors")
print(secondary_green)

print("\nWeak predictors")
print(weak_green)

In [ ]:
common_base = [
    "pr_mean",
    "pr_seasonality",
    "vpd_mean",
    "mrsos_mean",
    "lai_mean",
    "wue_mean",
]

common_sets = {
    "base": common_base,
    "base_plus_vpd_seasonality": common_base + ["vpd_seasonality"],
    "base_plus_RX1day": common_base + ["RX1day"],
    "base_plus_RX5day": common_base + ["RX5day"],
    "base_plus_vpd_seasonality_RX5day": common_base + ["vpd_seasonality", "RX5day"],
    "base_plus_both_extremes": common_base + ["RX1day", "RX5day"],
    "base_plus_mrso": common_base + ["RX5day", "vpd_seasonality", "mrso_mean"],
    "base_plus_clt": common_base + ["RX5day", "vpd_seasonality", "clt_mean"],
    "base_plus_tas": common_base + ["RX5day", "vpd_seasonality", "tas_mean"],
    "base_plus_rsds": common_base + ["RX5day", "vpd_seasonality", "rsds_mean"],
}

In [ ]:
blue_common_results = reg_analysis.compare_common_sets(
    X_blue, y_blue, groups_blue, common_sets, random_state=42
)

green_common_results = reg_analysis.compare_common_sets(
    X_green, y_green, groups_green, common_sets, random_state=42
)

print("Blue regime")
print(blue_common_results)
print()
print("Green regime")
print(green_common_results)

### Step 4: Main MLR and Fig. 3

In [ ]:
final_predictors = [
    "pr_mean",
    "pr_seasonality",
    "vpd_mean",
    "vpd_seasonality",
    "mrsos_mean",
    "clt_mean",
    "lai_mean",
    "wue_mean",
    "RX5day",
]

In [ ]:
blue_final_results, blue_final_coefs, blue_final_coef_summary, blue_final_perm, blue_final_perm_summary = (
    reg_analysis.repeated_grouped_importance_fixed_set(
        X_blue, y_blue, groups_blue,
        predictors=final_predictors,
        n_repeats=40,
        test_size=0.2,
        n_splits_inner=4,
        random_state=42,
        n_perm=20,
    )
)

green_final_results, green_final_coefs, green_final_coef_summary, green_final_perm, green_final_perm_summary = (
    reg_analysis.repeated_grouped_importance_fixed_set(
        X_green, y_green, groups_green,
        predictors=final_predictors,
        n_repeats=40,
        test_size=0.2,
        n_splits_inner=4,
        random_state=42,
        n_perm=20,
    )
)

In [ ]:
print("Blue model performance")
print(blue_final_results["test_r2"].describe())
print()
print("Blue coefficient summary")
print(blue_final_coef_summary)
print()
print("Blue permutation importance")
print(blue_final_perm_summary)

print("\n\nGreen model performance")
print(green_final_results["test_r2"].describe())
print()
print("Green coefficient summary")
print(green_final_coef_summary)
print()
print("Green permutation importance")
print(green_final_perm_summary)

In [ ]:
blue_final_summary = reg_analysis.summarize_importance_with_ci(blue_final_perm, blue_final_coefs)
green_final_summary = reg_analysis.summarize_importance_with_ci(green_final_perm, green_final_coefs)

print("Blue final summary")
print(blue_final_summary)

print("\nGreen final summary")
print(green_final_summary)

In [ ]:
per_model_long, perf_long, raw_outputs = reg_analysis.analyze_fixed_model_across_ensemble(
    ds_dict_change_sub=ds_dict_change_sub,
    scenario_key="ssp370_ff-historical",
    predictors=final_predictors,
    response="bgws_tran_mean",
    subdivisions=list(ds_dict_change_sub["ssp370_ff-historical"]["12 model ensemble mean"].subdivision.values),
    lat_bin=10,
    lon_bin=20,
    n_repeats=20,   
    test_size=0.2,
    n_splits_inner=4,
    random_state=42,
    n_perm=10,
)

In [ ]:
blue_agreement = reg_analysis.summarize_model_agreement(
    per_model_long,
    regime_name=list(ds_dict_change_sub["ssp370_ff-historical"]["12 model ensemble mean"].subdivision.values)[0]
)

green_agreement = reg_analysis.summarize_model_agreement(
    per_model_long,
    regime_name=list(ds_dict_change_sub["ssp370_ff-historical"]["12 model ensemble mean"].subdivision.values)[1]
)

In [ ]:
blue_agreement

In [ ]:
green_agreement

In [ ]:
blue_final_summary = reg_analysis.summarize_importance_with_iqr(blue_final_perm, blue_final_coefs)
green_final_summary = reg_analysis.summarize_importance_with_iqr(green_final_perm, green_final_coefs)

In [ ]:
display_names = {
    "RX5day": r"$\Delta$RX5day",
    "pr_mean": r"$\Delta P$",
    "pr_seasonality": r"$\Delta P_{\mathrm{seas}}$",
    "vpd_mean": r"$\Delta$VPD",
    "vpd_seasonality": r"$\Delta$VPD$_{\mathrm{seas}}$",
    "mrsos_mean": r"$\Delta$SM$_{\mathrm{surf}}$",
    "mrso_mean": r"$\Delta$SM",
    "clt_mean": r"$\Delta$CLT",
    "lai_mean": r"$\Delta$LAI",
    "wue_mean": r"$\Delta$WUE",
}

In [ ]:
blue_regime_name = list(ds_dict_change_sub["ssp370_ff-historical"]["12 model ensemble mean"].subdivision.values)[0]
green_regime_name = list(ds_dict_change_sub["ssp370_ff-historical"]["12 model ensemble mean"].subdivision.values)[1]

blue_context = reg_analysis.compute_clipped_standardized_context(df_blue, final_predictors, clip=1.0)
green_context = reg_analysis.compute_clipped_standardized_context(df_green, final_predictors, clip=1.0)

per_model_context_long = reg_analysis.compute_per_model_context_long(
    ds_dict_change_sub=ds_dict_change_sub,
    scenario_key="ssp370_ff-historical",
    predictors=final_predictors,
    clip=1.0,
)

In [ ]:
import importlib
importlib.reload(reg_analysis)

In [ ]:
fig, axes = reg_analysis.plot_driver_importance_two_panel(
    blue_summary=blue_final_summary.loc[final_predictors],
    green_summary=green_final_summary.loc[final_predictors],
    per_model_long=per_model_long,
    perf_long=perf_long,
    per_model_context_long=per_model_context_long,
    blue_regime_name=blue_regime_name,
    green_regime_name=green_regime_name,
    predictor_display_names=display_names,
    blue_context=blue_context,
    green_context=green_context,
    blue_r2=blue_final_results["test_r2"].mean(),
    green_r2=green_final_results["test_r2"].mean(),
    blue_n=len(df_blue),
    green_n=len(df_green),
    figsize=(17.2, 7.6),
    ticklabel_size=20,
    label_size=21,
    title_size=22,
    point_size=260,
    r2_threshold=0.3,
    ind_alpha=0.45,
    interval="iqr",

    savepath=None
)

plt.show()

### Step 5: Senstitvity tests (Table S3)

#### 5.1: RX1day sensitvity tests

In [ ]:
final_predictors_rx1 = [
    "pr_mean",
    "pr_seasonality",
    "vpd_mean",
    "vpd_seasonality",
    "mrsos_mean",
    "clt_mean",
    "lai_mean",
    "wue_mean",
    "RX1day",
]

In [ ]:
blue_rx1_results, blue_rx1_coefs, blue_rx1_coef_summary, blue_rx1_perm, blue_rx1_perm_summary = (
    reg_analysis.repeated_grouped_importance_fixed_set(
        X_blue, y_blue, groups_blue,
        predictors=final_predictors_rx1,
        n_repeats=40,
        test_size=0.2,
        n_splits_inner=4,
        random_state=42,
        n_perm=20,
    )
)

green_rx1_results, green_rx1_coefs, green_rx1_coef_summary, green_rx1_perm, green_rx1_perm_summary = (
    reg_analysis.repeated_grouped_importance_fixed_set(
        X_green, y_green, groups_green,
        predictors=final_predictors_rx1,
        n_repeats=40,
        test_size=0.2,
        n_splits_inner=4,
        random_state=42,
        n_perm=20,
    )
)

In [ ]:
comparison = pd.DataFrame({
    "blue_mean_r2": [
        blue_final_results["test_r2"].mean(),
        blue_rx1_results["test_r2"].mean(),
    ],
    "green_mean_r2": [
        green_final_results["test_r2"].mean(),
        green_rx1_results["test_r2"].mean(),
    ],
}, index=["RX5day_main", "RX1day_sensitivity"])

print(comparison)

In [ ]:
print("Blue RX1day sensitivity importance")
print(blue_rx1_perm_summary, blue_rx1_coef_summary)

print("\nGreen RX1day sensitivity importance")
print(green_rx1_perm_summary, green_rx1_coef_summary)

In [ ]:
blue_delta = blue_rx1_results["test_r2"].values - blue_final_results["test_r2"].values
green_delta = green_rx1_results["test_r2"].values - green_final_results["test_r2"].values

print("Blue ΔR² (RX1day - RX5day):")
print(pd.Series(blue_delta).describe())

print("\nGreen ΔR² (RX1day - RX5day):")
print(pd.Series(green_delta).describe())

In [ ]:
print(blue_rx1_results["test_r2"].values.mean()) 
print(green_rx1_results["test_r2"].values.mean())

In [ ]:
per_model_long_rx1, perf_long_rx1, raw_outputs_rx1 = reg_analysis.analyze_fixed_model_across_ensemble(
    ds_dict_change_sub=ds_dict_change_sub,
    scenario_key="ssp370_ff-historical",
    predictors=final_predictors_rx1,
    response="bgws_tran_mean",
    subdivisions=list(ds_dict_change_sub["ssp370_ff-historical"]["12 model ensemble mean"].subdivision.values),
    lat_bin=10,
    lon_bin=20,
    n_repeats=20,   # start smaller; raise later if needed
    test_size=0.2,
    n_splits_inner=4,
    random_state=42,
    n_perm=10,
)

In [ ]:
blue_agreement_rx1= reg_analysis.summarize_model_agreement(
    per_model_long_rx1,
    regime_name=list(ds_dict_change_sub["ssp370_ff-historical"]["12 model ensemble mean"].subdivision.values)[0]
)

green_agreement_rx1 = reg_analysis.summarize_model_agreement(
    per_model_long_rx1,
    regime_name=list(ds_dict_change_sub["ssp370_ff-historical"]["12 model ensemble mean"].subdivision.values)[1]
)

blue_final_summary_rx1 = reg_analysis.summarize_importance_with_iqr(blue_rx1_perm, blue_rx1_coefs)
green_final_summary_rx1 = reg_analysis.summarize_importance_with_iqr(green_rx1_perm, green_rx1_coefs)

In [ ]:
display_names = {
    "RX1day": r"$\Delta$RX1day",
    "pr_mean": r"$\Delta P$",
    "pr_seasonality": r"$\Delta P_{\mathrm{seas}}$",
    "vpd_mean": r"$\Delta$VPD",
    "vpd_seasonality": r"$\Delta$VPD$_{\mathrm{seas}}$",
    "mrsos_mean": r"$\Delta$SM$_{\mathrm{surf}}$",
    "mrso_mean": r"$\Delta$SM",
    "clt_mean": r"$\Delta$CLT",
    "lai_mean": r"$\Delta$LAI",
    "wue_mean": r"$\Delta$WUE",
}

blue_context_rx1 = reg_analysis.compute_clipped_standardized_context(df_blue, final_predictors_rx1, clip=1.0)
green_context_rx1 = reg_analysis.compute_clipped_standardized_context(df_green, final_predictors_rx1, clip=1.0)

per_model_context_long_rx1 = reg_analysis.compute_per_model_context_long(
    ds_dict_change_sub=ds_dict_change_sub,
    scenario_key="ssp370_ff-historical",
    predictors=final_predictors_rx1,
    clip=1.0,
)

In [ ]:
fig, axes = reg_analysis.plot_driver_importance_two_panel(
    blue_summary=masked_outputs[blue_regime_name]["summary_iqr"].loc[final_predictors],
    green_summary=masked_outputs[green_regime_name]["summary_iqr"].loc[final_predictors],
    per_model_long=per_model_long_masked,
    perf_long=perf_long_masked,
    per_model_context_long=per_model_context_long_masked,
    blue_regime_name=blue_regime_name,
    green_regime_name=green_regime_name,
    predictor_display_names=display_names,
    blue_context=blue_context_masked,
    green_context=green_context_masked,
    blue_r2=masked_outputs[blue_regime_name]["rep_results"]["test_r2"].mean(),
    green_r2=masked_outputs[green_regime_name]["rep_results"]["test_r2"].mean(),
    blue_n=len(df_blue_masked),
    green_n=len(df_green_masked),
    figsize=(17.2, 7.6),
    ticklabel_size=20,
    label_size=21,
    title_size=22,
    point_size=300,
    r2_threshold=0.3,
    ind_alpha=0.45,
    interval="iqr",
)

plt.show()

#### 5.2: Sensitivity test excluding uncertain grid cells and Figure S1

In [ ]:
scenario_key = "ssp370_ff-historical"
predictors_main = final_predictors   # use the same predictors as in your main analysis

mask_ds = reg_analysis.build_intermodel_sign_agreement_mask(
    ds_dict_change_sub=ds_dict_change_sub,
    scenario_key=scenario_key,
    var="bgws_tran_mean",
    min_agreement=0.80,   # try 0.75 too if 0.80 is too strict
)

confidence_mask = mask_ds["confidence_mask"]

print(mask_ds)

In [ ]:
mask_fraction_by_regime = confidence_mask.mean(dim=["lat", "lon"]).to_pandas()
valid_models_by_regime = mask_ds["n_valid_models"].median(dim=["lat", "lon"]).to_pandas()

print("Fraction of retained cells by subdivision:")
print(mask_fraction_by_regime)
print()
print("Median number of valid models by subdivision:")
print(valid_models_by_regime)

In [ ]:
ds_dict_change_sub_masked = reg_analysis.apply_subdivision_mask_to_scenario(
    ds_dict_change_sub=ds_dict_change_sub,
    scenario_key=scenario_key,
    mask_da=confidence_mask,
)

In [ ]:
combined_dict_diff['ssp370_ff-historical']['12 model ensemble mean']

In [ ]:
mask_ds = reg_analysis.build_intermodel_sign_agreement_mask(
    ds_dict_change_sub=combined_dict_diff,
    scenario_key=scenario_key,
    var="bgws_tran_mean",
    min_agreement=0.80,   # try 0.75 too if 0.80 is too strict
)

confidence_mask = mask_ds["confidence_mask"]

ds_dict_change_masked = reg_analysis.apply_subdivision_mask_to_scenario(
    ds_dict_change_sub=combined_dict_diff,
    scenario_key=scenario_key,
    mask_da=confidence_mask,
)

In [ ]:
ds_full = ds_dict_change_sub[scenario_key]["12 model ensemble mean"]
ds_mask = ds_dict_change_sub_masked[scenario_key]["12 model ensemble mean"]

In [ ]:
full_outputs = reg_analysis.run_fixed_model_for_subdivided_ds(
    ds=ds_full,
    predictors=predictors_main,
    response="bgws_tran_mean",
    lat_bin=10,
    lon_bin=20,
    n_repeats=40,
    test_size=0.2,
    n_splits_inner=4,
    random_state=42,
    n_perm=20,
)

masked_outputs = reg_analysis.run_fixed_model_for_subdivided_ds(
    ds=ds_mask,
    predictors=predictors_main,
    response="bgws_tran_mean",
    lat_bin=10,
    lon_bin=20,
    n_repeats=40,
    test_size=0.2,
    n_splits_inner=4,
    random_state=42,
    n_perm=20,
)

In [ ]:
rows = []
for subdivision in ds_full.subdivision.values:
    rows.append({
        "subdivision": subdivision,
        "n_full": len(full_outputs[subdivision]["df"]),
        "n_masked": len(masked_outputs[subdivision]["df"]),
        "blocks_full": full_outputs[subdivision]["df"]["spatial_block"].nunique(),
        "blocks_masked": masked_outputs[subdivision]["df"]["spatial_block"].nunique(),
        "mean_r2_full": full_outputs[subdivision]["rep_results"]["test_r2"].mean(),
        "mean_r2_masked": masked_outputs[subdivision]["rep_results"]["test_r2"].mean(),
        "delta_r2_masked_minus_full": (
            masked_outputs[subdivision]["rep_results"]["test_r2"].mean()
            - full_outputs[subdivision]["rep_results"]["test_r2"].mean()
        ),
    })

compare_perf = pd.DataFrame(rows)
print(compare_perf)

In [ ]:
for subdivision in ds_full.subdivision.values:
    print("\n" + "="*80)
    print(subdivision)
    print("- full")
    display(full_outputs[subdivision]["summary_iqr"][[
        "perm_median", "perm_q25", "perm_q75", "coef_median", "sign_consistency"
    ]].loc[predictors_main].sort_values("perm_median", ascending=False))

    print("- masked")
    display(masked_outputs[subdivision]["summary_iqr"][[
        "perm_median", "perm_q25", "perm_q75", "coef_median", "sign_consistency"
    ]].loc[predictors_main].sort_values("perm_median", ascending=False))

In [ ]:
per_model_long_masked, perf_long_masked, raw_outputs_masked = reg_analysis.analyze_fixed_model_across_ensemble(
    ds_dict_change_sub=ds_dict_change_sub_masked,
    scenario_key=scenario_key,
    predictors=predictors_main,
    response="bgws_tran_mean",
    subdivisions=list(ds_mask.subdivision.values),
    lat_bin=10,
    lon_bin=20,
    n_repeats=20,
    test_size=0.2,
    n_splits_inner=4,
    random_state=42,
    n_perm=10,
)

In [ ]:
blue_regime_name = list(ds_mask.subdivision.values)[0]
green_regime_name = list(ds_mask.subdivision.values)[1]

blue_agreement_masked = reg_analysis.summarize_model_agreement(per_model_long_masked, blue_regime_name)
green_agreement_masked = reg_analysis.summarize_model_agreement(per_model_long_masked, green_regime_name)

print("Masked blue agreement")
display(blue_agreement_masked)

print("Masked green agreement")
display(green_agreement_masked)

In [ ]:
# masked ensemble-mean dataset
ds_mask = ds_dict_change_sub_masked["ssp370_ff-historical"]["12 model ensemble mean"]

blue_regime_name = list(ds_mask.subdivision.values)[0]
green_regime_name = list(ds_mask.subdivision.values)[1]

# masked dataframes for context and n
df_blue_masked = reg_analysis.make_regime_dataframe_with_coords(
    ds_mask,
    subdivision=blue_regime_name,
    response="bgws_tran_mean",
    predictors=final_predictors,
)
df_green_masked = reg_analysis.make_regime_dataframe_with_coords(
    ds_mask,
    subdivision=green_regime_name,
    response="bgws_tran_mean",
    predictors=final_predictors,
)

blue_context_masked = reg_analysis.compute_clipped_standardized_context(df_blue_masked, final_predictors, clip=1.0)
green_context_masked = reg_analysis.compute_clipped_standardized_context(df_green_masked, final_predictors, clip=1.0)

per_model_context_long_masked = reg_analysis.compute_per_model_context_long(
    ds_dict_change_sub=ds_dict_change_sub_masked,
    scenario_key="ssp370_ff-historical",
    predictors=final_predictors,
    clip=1.0,
)

In [ ]:
fig, axes = reg_analysis.plot_driver_importance_two_panel(
    blue_summary=masked_outputs[blue_regime_name]["summary_iqr"].loc[final_predictors],
    green_summary=masked_outputs[green_regime_name]["summary_iqr"].loc[final_predictors],
    per_model_long=per_model_long_masked,
    perf_long=perf_long_masked,
    per_model_context_long=per_model_context_long_masked,
    blue_regime_name=blue_regime_name,
    green_regime_name=green_regime_name,
    predictor_display_names=display_names,
    blue_context=blue_context_masked,
    green_context=green_context_masked,
    blue_r2=masked_outputs[blue_regime_name]["rep_results"]["test_r2"].mean(),
    green_r2=masked_outputs[green_regime_name]["rep_results"]["test_r2"].mean(),
    blue_n=len(df_blue_masked),
    green_n=len(df_green_masked),
    figsize=(17.2, 7.6),
    ticklabel_size=20,
    label_size=21,
    title_size=22,
    point_size=300,
    r2_threshold=0.3,
    ind_alpha=0.45,
    interval="iqr",
)

plt.show()

In [ ]:
def plot_masked_vs_unmasked_map(
    cat,
    hist_bgws,
    title=None,
    figsize=(13.5, 5.8),
    fontsize=16,
    show_area_fraction=True,
):
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    from matplotlib.colors import ListedColormap, BoundaryNorm
    from matplotlib.patches import Patch
    import cartopy.crs as ccrs
    import xarray as xr

    lat_name = "lat" if "lat" in cat.coords else "latitude"

    # Original cat:
    # 0 = invalid/ocean
    # 1 = retained / high confidence
    # 2 = masked out
    #
    # New cat_regime:
    # 0 = invalid/ocean
    # 1 = high confidence + historical blue water regime
    # 2 = high confidence + historical green water regime
    # 3 = masked out

    blue_regime = hist_bgws > 0
    green_regime = hist_bgws < 0

    cat_regime = xr.full_like(cat, fill_value=0, dtype=int)
    cat_regime = xr.where((cat == 1) & blue_regime, 1, cat_regime)
    cat_regime = xr.where((cat == 1) & green_regime, 2, cat_regime)
    cat_regime = xr.where(cat == 2, 3, cat_regime)

    # Colors
    cmap = ListedColormap([
        "white",      # invalid / ocean
        "#2f7ed8",    # high confidence, historical blue water regime
        "#137a13",    # high confidence, historical green water regime
        "0.82",       # masked out
    ])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

    # --- n counts ---
    n_blue = int((cat_regime == 1).sum().values.item())
    n_green = int((cat_regime == 2).sum().values.item())
    n_high = n_blue + n_green
    n_masked = int((cat_regime == 3).sum().values.item())

    # --- area-weighted fractions ---
    lat = cat[lat_name]
    weights_1d = np.cos(np.deg2rad(lat))
    weights_2d = weights_1d.broadcast_like(cat)

    valid_domain = cat_regime.isin([1, 2, 3])

    area_blue = float(
        weights_2d.where(cat_regime == 1).sum(skipna=True)
        / weights_2d.where(valid_domain).sum(skipna=True)
    )
    area_green = float(
        weights_2d.where(cat_regime == 2).sum(skipna=True)
        / weights_2d.where(valid_domain).sum(skipna=True)
    )
    area_high = area_blue + area_green
    area_masked = float(
        weights_2d.where(cat_regime == 3).sum(skipna=True)
        / weights_2d.where(valid_domain).sum(skipna=True)
    )

    plt.rcParams.update({
        "font.size": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize,
        "ytick.labelsize": fontsize,
        "legend.fontsize": fontsize - 1,
    })

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

    ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

    cat_regime.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        norm=norm,
        add_colorbar=False,
        rasterized=True,
    )

    ax.coastlines(linewidth=0.8, zorder=5)

    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.5,
        color="gray",
        alpha=0.5,
        linestyle="--",
        zorder=6,
    )
    gl.xlocator = mticker.FixedLocator([-120, -60, 0, 60, 120])
    gl.ylocator = mticker.FixedLocator([-40, 0, 40, 80])
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = True
    gl.bottom_labels = True
    gl.xlabel_style = {"size": fontsize}
    gl.ylabel_style = {"size": fontsize}

    ax.set_title("" if title is None else title)
    ax.set_xlabel("")
    ax.set_ylabel("")

    # --- text box with n ---
    if show_area_fraction:
        textstr = (
            f"High confidence\n"
            f"n = {n_high:,} ({area_high*100:.1f}%)\n\n"
            f"Blue regime:\nn = {n_blue:,} ({area_blue*100:.1f}%)\n"
            f"Green regime:\nn = {n_green:,} ({area_green*100:.1f}%)\n\n"
            f"Masked out:\n"
            f"n = {n_masked:,} ({area_masked*100:.1f}%)"
        )
    else:
        textstr = (
            f"High confidence: n = {n_high:,}\n"
            f"Blue regime: n = {n_blue:,}\n"
            f"Green regime: n = {n_green:,}\n\n"
            f"Masked out: n = {n_masked:,}"
        )


    # --- custom text box with individually styled lines ---
    
    x0, y0 = 0.02, 0.05          # position in axes coordinates
    line_gap = 0.055             # vertical spacing between lines
    box_width = 0.4
    box_height = 0.51
    
    # background box
    box = dict(
        facecolor="white",
        edgecolor="black",
        boxstyle="round,pad=0.35",
        alpha=0.85,
        linewidth=0.8,
    )

    from matplotlib.patches import FancyBboxPatch
    
    rect = FancyBboxPatch(
        (0.035, 0.02),
        0.15,
        0.5,
        transform=ax.transAxes,
        boxstyle="round,pad=0.01",
        facecolor="white",
        edgecolor="black",
        linewidth=0.8,
        alpha=0.85,
        zorder=19,
    )
    ax.add_patch(rect)
        
    # text lines: y positions are manual
    lines = [
        {
            "text": "High confidence:",
            "fontsize": fontsize - 2,
            "fontweight": "bold",
            "color": "black",
        },
        {
            "text": f"n = {n_high:,} ({area_high*100:.1f}%)",
            "fontsize": fontsize - 3,
            "fontweight": "normal",
            "color": "black",
        },
        {
            "text": "Blue regime:",
            "fontsize": fontsize - 3,
            "fontweight": "bold",
            "color": "#2f7ed8",
        },
        {
            "text": f"n = {n_blue:,} ({area_blue*100:.1f}%)",
            "fontsize": fontsize - 3,
            "fontweight": "normal",
            "color": "black",
        },
        {
            "text": "Green regime:",
            "fontsize": fontsize - 3,
            "fontweight": "bold",
            "color": "#137a13",
        },
        {
            "text": f"n = {n_green:,} ({area_green*100:.1f}%)",
            "fontsize": fontsize - 3,
            "fontweight": "normal",
            "color": "black",
        },
         {
            "text": f"",
            "fontsize": fontsize - 3,
            "fontweight": "normal",
            "color": "black",
        },
        {
            "text": "Masked out:",
            "fontsize": fontsize - 2,
            "fontweight": "bold",
            "color": "black",
        },
        {
            "text": f"n = {n_masked:,} ({area_masked*100:.1f}%)",
            "fontsize": fontsize - 3,
            "fontweight": "normal",
            "color": "black",
        },
    ]
    
    # starting y inside the box
    start_y = y0 + box_height - 0.045
    
    for i, line in enumerate(lines):
        ax.text(
            x0 + 0.012,
            start_y - i * line_gap,
            line["text"],
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=line["fontsize"],
            fontweight=line["fontweight"],
            color=line["color"],
            zorder=21,
        )

    legend_elements = [
        Patch(
            facecolor="#2f7ed8",
            edgecolor="black",
            linewidth=0.8,
            label="High confidence (historical blue water regime)"
        ),
        Patch(
            facecolor="#137a13",
            edgecolor="black",
            linewidth=0.8,
            label="High confidence (historical green water regime)"
        ),
        Patch(
            facecolor="0.82",
            edgecolor="black",
            linewidth=0.8,
            label="Masked out"
        ),
    ]

    ax.legend(
        handles=legend_elements,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.10),
        ncol=2,
        frameon=False,
        handlelength=1.5,
        columnspacing=1.5,
    )

    fig.subplots_adjust(left=0.03, right=0.99, top=0.96, bottom=0.23)
    plt.show()

    return fig, ax, cat_regime

In [ ]:
hist_bgws = combined_dict["historical"]["12 model ensemble mean"].bgws_tran_mean

fig, ax, cat_regime = plot_masked_vs_unmasked_map(
    cat=cat,
    hist_bgws=hist_bgws,
    title="",
    figsize=(13.5, 5.8),
    fontsize=16,
    show_area_fraction=True,
)

In [ ]:
def add_high_confidence_regime_map_panel(
    ax,
    cat,
    hist_bgws,
    fontsize=16,
    panel_label="(c)",
    show_textbox=True,
):
    import numpy as np
    import xarray as xr
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    from matplotlib.colors import ListedColormap, BoundaryNorm
    from matplotlib.patches import Patch, FancyBboxPatch
    import cartopy.crs as ccrs

    lat_name = "lat" if "lat" in cat.coords else "latitude"

    # Original cat:
    # 0 = invalid/ocean
    # 1 = retained / high confidence
    # 2 = masked out
    #
    # New cat_regime:
    # 0 = invalid/ocean
    # 1 = high confidence + historical blue water regime
    # 2 = high confidence + historical green water regime
    # 3 = masked out

    blue_regime = hist_bgws > 0
    green_regime = hist_bgws < 0

    cat_regime = xr.full_like(cat, fill_value=0, dtype=int)
    cat_regime = xr.where((cat == 1) & blue_regime, 1, cat_regime)
    cat_regime = xr.where((cat == 1) & green_regime, 2, cat_regime)
    cat_regime = xr.where(cat == 2, 3, cat_regime)

    color_blue = "#2f7ed8"
    color_green = "#137a13"
    color_masked = "0.82"

    cmap = ListedColormap([
        "white",       # invalid / ocean
        color_blue,    # high confidence, historical blue water regime
        color_green,   # high confidence, historical green water regime
        color_masked,  # masked out
    ])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

    # Counts
    n_blue = int((cat_regime == 1).sum().values.item())
    n_green = int((cat_regime == 2).sum().values.item())
    n_high = n_blue + n_green
    n_masked = int((cat_regime == 3).sum().values.item())

    # Area-weighted fractions
    lat = cat[lat_name]
    weights_1d = np.cos(np.deg2rad(lat))
    weights_2d = weights_1d.broadcast_like(cat)

    valid_domain = cat_regime.isin([1, 2, 3])

    area_blue = float(
        weights_2d.where(cat_regime == 1).sum(skipna=True)
        / weights_2d.where(valid_domain).sum(skipna=True)
    )
    area_green = float(
        weights_2d.where(cat_regime == 2).sum(skipna=True)
        / weights_2d.where(valid_domain).sum(skipna=True)
    )
    area_high = area_blue + area_green
    area_masked = float(
        weights_2d.where(cat_regime == 3).sum(skipna=True)
        / weights_2d.where(valid_domain).sum(skipna=True)
    )

    # Map
    ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

    cat_regime.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        norm=norm,
        add_colorbar=False,
        rasterized=True,
    )

    ax.coastlines(linewidth=0.8, zorder=5)

    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.5,
        color="gray",
        alpha=0.5,
        linestyle="--",
        zorder=6,
    )
    gl.xlocator = mticker.FixedLocator([-120, -60, 0, 60, 120])
    gl.ylocator = mticker.FixedLocator([-40, 0, 40, 80])
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = True
    gl.bottom_labels = True
    gl.xlabel_style = {"size": fontsize}
    gl.ylabel_style = {"size": fontsize}

    ax.set_title("")
    ax.set_xlabel("")
    ax.set_ylabel("")

    # Panel label
    ax.text(
        -0.04, 1.03,
        panel_label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=fontsize + 8,
        fontweight="bold",
        zorder=30,
    )

    # Textbox
    if show_textbox:
        x0, y0 = 0.035, 0.03
        box_width = 0.155
        box_height = 0.50
        line_gap = 0.061

        rect = FancyBboxPatch(
            (x0, y0),
            box_width,
            box_height,
            transform=ax.transAxes,
            boxstyle="round,pad=0.01",
            facecolor="white",
            edgecolor="black",
            linewidth=0.8,
            alpha=0.88,
            zorder=19,
        )
        ax.add_patch(rect)

        lines = [
            ("High confidence:", fontsize - 2, "bold", "black"),
            (f"n = {n_high:,} ({area_high*100:.1f}%)", fontsize - 3, "normal", "black"),
            ("", fontsize - 3, "normal", "black"),
            ("Blue regime:", fontsize - 3, "bold", color_blue),
            (f"n = {n_blue:,} ({area_blue*100:.1f}%)", fontsize - 3, "normal", "black"),
            ("Green regime:", fontsize - 3, "bold", color_green),
            (f"n = {n_green:,} ({area_green*100:.1f}%)", fontsize - 3, "normal", "black"),
            ("", fontsize - 3, "normal", "black"),
            ("Masked out:", fontsize - 2, "bold", "black"),
            (f"n = {n_masked:,} ({area_masked*100:.1f}%)", fontsize - 3, "normal", "black"),
        ]

        start_y = y0 + box_height - 0.035

        for i, (txt, fs, fw, col) in enumerate(lines):
            ax.text(
                x0 + 0.012,
                start_y - i * line_gap,
                txt,
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=fs,
                fontweight=fw,
                color=col,
                zorder=21,
            )

    legend_elements = [
        Patch(
            facecolor=color_blue,
            edgecolor="black",
            linewidth=0.8,
            label="High confidence (historical blue water regime)",
        ),
        Patch(
            facecolor=color_green,
            edgecolor="black",
            linewidth=0.8,
            label="High confidence (historical green water regime)",
        ),
        Patch(
            facecolor=color_masked,
            edgecolor="black",
            linewidth=0.8,
            label="Masked out",
        ),
    ]

    ax.legend(
        handles=legend_elements,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
        ncol=2,
        frameon=False,
        handlelength=1.5,
        columnspacing=1.5,
        fontsize=fontsize - 1,
    )

    return cat_regime

#### 5.3: Land use variables

In [ ]:
#### Load land use data if you want to reproduce Table S3 

_, masked_ds_dict_diff_land_use, _ = comp_stats.selected_driver_landuse_period_stats(hydro_mask=hydro_mask, landuse_vars=("treeFrac", "cropFrac"))

In [ ]:
scenario_key = 'ssp370_ff-historical'
ensemble_mean_key = '9 model ensemble mean'

In [ ]:
xr.corr(masked_ds_dict_diff_land_use[scenario_key][ensemble_mean_key].lai_mean, masked_ds_dict_diff_land_use[scenario_key][ensemble_mean_key].treeFrac_mean).values

In [ ]:
import numpy as np
from scipy.stats import spearmanr

def xr_spearmanr(da1, da2, mask=None):
    """
    Spearman rank correlation between two xarray DataArrays.
    Optionally restricted to mask=True.
    """
    valid = np.isfinite(da1) & np.isfinite(da2)

    if mask is not None:
        valid = valid & mask

    x = da1.where(valid).values.ravel()
    y = da2.where(valid).values.ravel()

    keep = np.isfinite(x) & np.isfinite(y)

    rho, pval = spearmanr(x[keep], y[keep])

    return rho, pval, keep.sum()

In [ ]:
import numpy as np
import xarray as xr
from scipy.stats import rankdata

def weighted_corr(x, y, w):
    """Weighted Pearson correlation."""
    w = np.asarray(w)
    x = np.asarray(x)
    y = np.asarray(y)

    w = w / np.sum(w)

    mx = np.sum(w * x)
    my = np.sum(w * y)

    cov_xy = np.sum(w * (x - mx) * (y - my))
    var_x = np.sum(w * (x - mx)**2)
    var_y = np.sum(w * (y - my)**2)

    return cov_xy / np.sqrt(var_x * var_y)


def xr_weighted_spearmanr(da1, da2, mask=None):
    """
    Area-weighted Spearman correlation between two xarray DataArrays.
    Uses cosine-latitude weights.
    """
    lat_name = "lat" if "lat" in da1.coords else "latitude"

    valid = np.isfinite(da1) & np.isfinite(da2)

    if mask is not None:
        valid = valid & mask

    weights_1d = np.cos(np.deg2rad(da1[lat_name]))
    weights_2d = weights_1d.broadcast_like(da1)

    x = da1.where(valid).values.ravel()
    y = da2.where(valid).values.ravel()
    w = weights_2d.where(valid).values.ravel()

    keep = np.isfinite(x) & np.isfinite(y) & np.isfinite(w) & (w > 0)

    x_rank = rankdata(x[keep])
    y_rank = rankdata(y[keep])

    rho_w = weighted_corr(x_rank, y_rank, w[keep])

    return rho_w, keep.sum()

In [ ]:
rho, pval, n = xr_spearmanr(masked_ds_dict_diff_land_use[scenario_key][ensemble_mean_key].lai_mean, masked_ds_dict_diff_land_use[scenario_key][ensemble_mean_key].treeFrac_mean)

print(f"Spearman rho = {rho:.3f}")
print(f"p-value = {pval:.3e}")
print(f"n = {n}")

In [ ]:
rho_w, n = xr_weighted_spearmanr(masked_ds_dict_diff_land_use[scenario_key][ensemble_mean_key].lai_mean, masked_ds_dict_diff_land_use[scenario_key][ensemble_mean_key].treeFrac_mean)

print(f"Area-weighted Spearman rho = {rho_w:.3f}")
print(f"n = {n}")

In [ ]:
ds_dict_change_sub_landuse = pro_dat.subdivide_ds_dict(
    ds_dict_base=combined_dict["historical"],
    ds_dict_change=masked_ds_dict_diff_land_use["ssp370_ff-historical"],
    base_id="historical",
    change_id="ssp370_ff-historical",
    variable="bgws_tran_mean",
    base_key="12 model ensemble mean",
)

In [ ]:
print(ds_dict_change_sub_landuse['ssp370_ff-historical']['9 model ensemble mean'])
print(ds_dict_change_sub_landuse['ssp370_ff-historical']['9 model ensemble mean'].subdivision.values)

In [ ]:
base_predictors_9m = [
    "pr_mean",
    "pr_seasonality",
    "vpd_mean",
    "vpd_seasonality",
    "mrsos_mean",
    "lai_mean",
    "clt_mean",
    "wue_mean",
    "RX5day",
]

In [ ]:
tree_predictors_9m = base_predictors_9m + ["treeFrac_mean"]

tree_crop_predictors_9m = base_predictors_9m + ["treeFrac_mean", "cropFrac_mean"]

In [ ]:
response = "bgws_tran_mean"

ds = ds_dict_change_sub_landuse[scenario_key][ensemble_mean_key]
subdivision_names = list(ds.subdivision.values)

df_blue_9m = reg_analysis.make_regime_dataframe_with_coords(
    ds, subdivision=subdivision_names[0],
    response=response,
    predictors=tree_crop_predictors_9m if "cropFrac_mean" in ds.data_vars else tree_predictors_9m,
)

df_green_9m = reg_analysis.make_regime_dataframe_with_coords(
    ds, subdivision=subdivision_names[1],
    response=response,
    predictors=tree_crop_predictors_9m if "cropFrac_mean" in ds.data_vars else tree_predictors_9m,
)

df_blue_9m = reg_analysis.add_spatial_blocks(df_blue_9m, lat_bin=10, lon_bin=20)
df_green_9m = reg_analysis.add_spatial_blocks(df_green_9m, lat_bin=10, lon_bin=20)

In [ ]:
all_predictors_9m = [c for c in df_blue_9m.columns if c not in ["lat", "lon", "lat_block", "lon_block", "spatial_block", response]]

X_blue_9m = df_blue_9m[all_predictors_9m].copy()
y_blue_9m = df_blue_9m[response].copy()
groups_blue_9m = df_blue_9m["spatial_block"].copy()

X_green_9m = df_green_9m[all_predictors_9m].copy()
y_green_9m = df_green_9m[response].copy()
groups_green_9m = df_green_9m["spatial_block"].copy()

In [ ]:
common_sets_9m = {
    "baseline_9m": base_predictors_9m,
    "plus_treeFrac_9m": tree_predictors_9m,
    "plus_tree_crop_9m": tree_crop_predictors_9m,
}

In [ ]:
blue_landuse_compare = reg_analysis.compare_common_sets(
    X_blue_9m, y_blue_9m, groups_blue_9m, common_sets_9m, random_state=42
)

green_landuse_compare = reg_analysis.compare_common_sets(
    X_green_9m, y_green_9m, groups_green_9m, common_sets_9m, random_state=42
)

print("Blue land-use sensitivity")
print(blue_landuse_compare)
print()

print("Green land-use sensitivity")
print(green_landuse_compare)

In [ ]:
blue_9m_results, blue_9m_coefs, blue_9m_coef_summary, blue_9m_perm, blue_9m_perm_summary = (
    reg_analysis.repeated_grouped_importance_fixed_set(
        X_blue_9m, y_blue_9m, groups_blue_9m,
        predictors=base_predictors_9m,
        n_repeats=40,
        test_size=0.2,
        n_splits_inner=4,
        random_state=42,
        n_perm=20,
    )
)

green_9m_results, green_9m_coefs, green_9m_coef_summary, green_9m_perm, green_9m_perm_summary = (
    reg_analysis.repeated_grouped_importance_fixed_set(
        X_green_9m, y_green_9m, groups_green_9m,
        predictors=base_predictors_9m,
        n_repeats=40,
        test_size=0.2,
        n_splits_inner=4,
        random_state=42,
        n_perm=20,
    )
)

In [ ]:
blue_9m_summary = reg_analysis.summarize_importance_with_ci(blue_9m_perm, blue_9m_coefs)
green_9m_summary = reg_analysis.summarize_importance_with_ci(green_9m_perm, green_9m_coefs)

print("Blue 9 models")
print(blue_9m_summary)

print("\nGreen 9 models")
print(green_9m_summary)

In [ ]:
tree_model_predictors = tree_predictors_9m

blue_tree_results, blue_tree_coefs, blue_tree_coef_summary, blue_tree_perm, blue_tree_perm_summary = (
    reg_analysis.repeated_grouped_importance_fixed_set(
        X_blue_9m, y_blue_9m, groups_blue_9m,
        predictors=tree_model_predictors,
        n_repeats=40,
        test_size=0.2,
        n_splits_inner=4,
        random_state=42,
        n_perm=20,
    )
)

green_tree_results, green_tree_coefs, green_tree_coef_summary, green_tree_perm, green_tree_perm_summary = (
    reg_analysis.repeated_grouped_importance_fixed_set(
        X_green_9m, y_green_9m, groups_green_9m,
        predictors=tree_model_predictors,
        n_repeats=40,
        test_size=0.2,
        n_splits_inner=4,
        random_state=42,
        n_perm=20,
    )
)

In [ ]:
blue_tree_summary = reg_analysis.summarize_importance_with_ci(blue_tree_perm, blue_tree_coefs)
green_tree_summary = reg_analysis.summarize_importance_with_ci(green_tree_perm, green_tree_coefs)

print("Blue + treeFrac")
print(blue_tree_summary)

print("\nGreen + treeFrac")
print(green_tree_summary)

In [ ]:
tree_crop_model_predictors = [
    "pr_mean",
    "pr_seasonality",
    "vpd_mean",
    "vpd_seasonality",
    "mrsos_mean",
    "lai_mean",
    "wue_mean",
    "clt_mean",
    "RX5day",
    "treeFrac_mean",
    "cropFrac_mean",
]

blue_tree_crop_results, blue_tree_crop_coefs, blue_tree_crop_coef_summary, blue_tree_crop_perm, blue_tree_crop_perm_summary = (
    reg_analysis.repeated_grouped_importance_fixed_set(
        X_blue_9m, y_blue_9m, groups_blue_9m,
        predictors=tree_crop_model_predictors,
        n_repeats=40,
        test_size=0.2,
        n_splits_inner=4,
        random_state=42,
        n_perm=20,
    )
)

green_tree_crop_results, green_tree_crop_coefs, green_tree_crop_coef_summary, green_tree_crop_perm, green_tree_crop_perm_summary = (
    reg_analysis.repeated_grouped_importance_fixed_set(
        X_green_9m, y_green_9m, groups_green_9m,
        predictors=tree_crop_model_predictors,
        n_repeats=40,
        test_size=0.2,
        n_splits_inner=4,
        random_state=42,
        n_perm=20,
    )
)

In [ ]:
blue_tree_crop_summary = reg_analysis.summarize_importance_with_ci(
    blue_tree_crop_perm, blue_tree_crop_coefs
)

green_tree_crop_summary = reg_analysis.summarize_importance_with_ci(
    green_tree_crop_perm, green_tree_crop_coefs
)

print("Blue + treeFrac + cropFrac")
print(blue_tree_crop_summary)

print("\nGreen + treeFrac + cropFrac")
print(green_tree_crop_summary)

In [ ]:
display_names_landuse = {
    "RX5day": r"$\Delta$RX5day",
    "pr_mean": r"$\Delta P$",
    "pr_seasonality": r"$\Delta P_{\mathrm{season}}$",
    "vpd_mean": r"$\Delta$VPD",
    "vpd_seasonality": r"$\Delta$VPD$_{\mathrm{season}}$",
    "mrsos_mean": r"$\Delta$SM$_{\mathrm{surf}}$",
    "lai_mean": r"$\Delta$LAI",
    "clt_mean": r"$\Delta$CLT",
    "wue_mean": r"$\Delta$WUE",
    "treeFrac_mean": r"$\Delta$TreeFrac",
    "cropFrac_mean": r"$\Delta$CropFrac",
}

In [ ]:
scenario_key = "ssp370_ff-historical"
ensemble_mean_key = "9 model ensemble mean"

m9_outputs = reg_analysis.run_landuse_driver_figure(
    ds_dict_change_sub_landuse=ds_dict_change_sub_landuse,
    scenario_key=scenario_key,
    ensemble_mean_key=ensemble_mean_key,
    predictors=base_predictors_9m,
    predictor_display_names=display_names_landuse,
    response="bgws_tran_mean",
    lat_bin=10,
    lon_bin=20,
    n_repeats_main=40,
    n_repeats_esm=20,
    test_size=0.2,
    n_splits_inner=4,
    random_state=42,
    n_perm_main=20,
    n_perm_esm=10,
    r2_threshold=0.3,
    figsize=(17.2, 8.2),
    ticklabel_size=20,
    label_size=21,
    title_size=22,
    point_size=300,
    ind_alpha=0.45,
    interval="iqr",
)

plt.show()

In [ ]:
scenario_key = "ssp370_ff-historical"
ensemble_mean_key = "9 model ensemble mean"

tree_outputs = reg_analysis.run_landuse_driver_figure(
    ds_dict_change_sub_landuse=ds_dict_change_sub_landuse,
    scenario_key=scenario_key,
    ensemble_mean_key=ensemble_mean_key,
    predictors=tree_predictors_9m,
    predictor_display_names=display_names_landuse,
    response="bgws_tran_mean",
    lat_bin=10,
    lon_bin=20,
    n_repeats_main=40,
    n_repeats_esm=20,
    test_size=0.2,
    n_splits_inner=4,
    random_state=42,
    n_perm_main=20,
    n_perm_esm=10,
    r2_threshold=0.3,
    figsize=(17.2, 8.2),
    ticklabel_size=20,
    label_size=21,
    title_size=22,
    point_size=300,
    ind_alpha=0.45,
    interval="iqr",
)

plt.show()

In [ ]:
tree_crop_outputs = reg_analysis.run_landuse_driver_figure(
    ds_dict_change_sub_landuse=ds_dict_change_sub_landuse,
    scenario_key=scenario_key,
    ensemble_mean_key=ensemble_mean_key,
    predictors=tree_crop_predictors_9m,
    predictor_display_names=display_names_landuse,
    response="bgws_tran_mean",
    lat_bin=10,
    lon_bin=20,
    n_repeats_main=40,
    n_repeats_esm=20,
    test_size=0.2,
    n_splits_inner=4,
    random_state=42,
    n_perm_main=20,
    n_perm_esm=10,
    r2_threshold=0.3,
    figsize=(17.2, 8.8),   # slightly taller for 10 predictors
    ticklabel_size=20,
    label_size=21,
    title_size=22,
    point_size=300,
    ind_alpha=0.45,
    interval="iqr",
)

plt.show()

In [ ]:
print("Tree-only: blue")
display(tree_outputs["blue_summary"].loc[tree_predictors_9m].sort_values("perm_median", ascending=False))

print("Tree-only: green")
display(tree_outputs["green_summary"].loc[tree_predictors_9m].sort_values("perm_median", ascending=False))

print("Tree+crop: blue")
display(tree_crop_outputs["blue_summary"].loc[tree_crop_predictors_9m].sort_values("perm_median", ascending=False))

print("Tree+crop: green")
display(tree_crop_outputs["green_summary"].loc[tree_crop_predictors_9m].sort_values("perm_median", ascending=False))

### Step 6: RF and shap (Table S5 and Figure 11)

In [ ]:
rf_grid_small = {
    "n_estimators": [200],
    "max_depth": [8, 16],
    "min_samples_leaf": [3, 5],
    "min_samples_split": [2, 10],
    "max_features": [0.5, 1.0],
}

In [ ]:
blue_rf_params, blue_rf_cv = reg_analysis.tune_grouped_rf_once(
    X_blue, y_blue, groups_blue,
    predictors=final_predictors,
    n_splits_inner=4,
    random_state=42,
    param_grid=rf_grid_small,
    n_jobs=1,
)

In [ ]:
green_rf_params, green_rf_cv = reg_analysis.tune_grouped_rf_once(
    X_green, y_green, groups_green,
    predictors=final_predictors,
    n_splits_inner=4,
    random_state=42,
    param_grid=rf_grid_small,
    n_jobs=1,
)

print("Blue best params:", blue_rf_params)
print("Green best params:", green_rf_params)

In [ ]:
rf_blue_results, rf_blue_perm, rf_blue_perm_summary, rf_blue_shap, rf_blue_shap_summary, rf_blue_shap_values, rf_blue_shap_features = (
    reg_analysis.repeated_grouped_rf_shap_fixed_params(
        X_blue,
        y_blue,
        groups_blue,
        predictors=final_predictors,
        rf_params=blue_rf_params,
        n_repeats=10,
        test_size=0.2,
        random_state=42,
        n_perm=5,
        max_shap_samples=300,
        store_all_shap=False,
    )
)

In [ ]:
rf_green_results, rf_green_perm, rf_green_perm_summary, rf_green_shap, rf_green_shap_summary, rf_green_shap_values, rf_green_shap_features = (
    reg_analysis.repeated_grouped_rf_shap_fixed_params(
        X_green,
        y_green,
        groups_green,
        predictors=final_predictors,
        rf_params=green_rf_params,
        n_repeats=10,
        test_size=0.2,
        random_state=42,
        n_perm=5,
        max_shap_samples=300,
        store_all_shap=False,
    )
)

In [ ]:
blue_rep = reg_analysis.choose_representative_repeat(rf_blue_results)
print("Representative blue repeat:", blue_rep)

blue_shap_out = reg_analysis.fit_rf_one_grouped_repeat_with_shap(
    X_blue,
    y_blue,
    groups_blue,
    predictors=final_predictors,
    rf_params=blue_rf_params,
    repeat_to_use=blue_rep,
    n_repeats_total=10,   # must match the repeated RF run
    test_size=0.2,
    random_state=42,
    max_shap_samples=500,
)

print("Blue representative repeat test R2:", blue_shap_out["test_r2"])

In [ ]:
display_names_rf = {
    "RX5day": r"$\Delta$RX5day",
    "pr_mean": r"$\Delta P$",
    "pr_seasonality": r"$\Delta P_{\mathrm{seas}}$",
    "vpd_mean": r"$\Delta$VPD",
    "vpd_seasonality": r"$\Delta$VPD$_{\mathrm{seas}}$",
    "mrsos_mean": r"$\Delta$SM$_{\mathrm{surf}}$",
    "lai_mean": r"$\Delta$LAI",
    "wue_mean": r"$\Delta$WUE",
    "clt_mean": r"$\Delta$CLT",
}

feature_names_rf = [display_names_rf.get(p, p) for p in final_predictors]

In [ ]:
reg_analysis.plot_rf_shap_beeswarm(
    shap_df=blue_shap_out["shap_values"],
    X_shap_df=blue_shap_out["X_shap"],
    predictors=final_predictors,
    display_names=display_names_landuse,
    title="Blue water regime",
    max_display=len(final_predictors),
    figsize=(8, 5.5),
)

In [ ]:
green_rep = reg_analysis.choose_representative_repeat(rf_green_results)
print("Representative green repeat:", green_rep)

green_shap_out = reg_analysis.fit_rf_one_grouped_repeat_with_shap(
    X_green,
    y_green,
    groups_green,
    predictors=final_predictors,
    rf_params=green_rf_params,
    repeat_to_use=green_rep,
    n_repeats_total=10,   # must match the repeated RF run
    test_size=0.2,
    random_state=42,
    max_shap_samples=500,
)

print("Green representative repeat test R2:", green_shap_out["test_r2"])

In [ ]:
reg_analysis.plot_rf_shap_beeswarm(
    shap_df=green_shap_out["shap_values"],
    X_shap_df=green_shap_out["X_shap"],
    predictors=final_predictors,
    display_names=display_names_rf,
    title="Green water regime",
    max_display=len(final_predictors),
    figsize=(8, 5.5),
)

In [ ]:
print(rf_blue_results[["train_r2", "test_r2", "overfit_gap"]].describe())
print(rf_green_results[["train_r2", "test_r2", "overfit_gap"]].describe())

In [ ]:
compare_blue = pd.DataFrame({
    "linear_perm_median": blue_final_summary.loc[final_predictors, "perm_median"],
    "rf_perm_median": rf_blue_perm_summary.loc[final_predictors, "perm_median"],
    "rf_shap_median": rf_blue_shap_summary.loc[final_predictors, "shap_median"],
}).sort_values("rf_shap_median", ascending=False)

compare_green = pd.DataFrame({
    "linear_perm_median": green_final_summary.loc[final_predictors, "perm_median"],
    "rf_perm_median": rf_green_perm_summary.loc[final_predictors, "perm_median"],
    "rf_shap_median": rf_green_shap_summary.loc[final_predictors, "shap_median"],
}).sort_values("rf_shap_median", ascending=False)

print("Blue comparison")
display(compare_blue)

print("Green comparison")
display(compare_green)

In [ ]:
blue_shap_importance_for_labels = rf_blue_shap_summary["shap_median"].copy()
green_shap_importance_for_labels = rf_green_shap_summary["shap_median"].copy()

In [ ]:
fig, axes = reg_analysis.plot_rf_shap_two_panel_composite_row(
    blue_shap_out=blue_shap_out,
    green_shap_out=green_shap_out,
    blue_importance=rf_blue_shap_summary["shap_median"],
    green_importance=rf_green_shap_summary["shap_median"],
    display_names=display_names_rf,
    blue_r2=rf_blue_results["test_r2"].mean(),
    green_r2=rf_green_results["test_r2"].mean(),
    blue_n=len(df_blue),
    green_n=len(df_green),
    shap_figsize=(7.5, 2.6),
    shap_dpi=220,
    final_figsize=(15, 5.4),
    cmap_name="coolwarm",
    cbar_label="Normalised predictor change",
)

plt.show()